# 01 · Verificación de datos crudos
Valida que los 5 archivos del Yelp Open Dataset (muestreados) estén en el Volume
y hace una primera exploración de esquemas. **No transforma nada** — eso es trabajo
del pipeline (E7).

In [0]:
RAW = "/Volumes/workspace/yelp_bronze/raw"
ESPERADOS = ["business", "review", "user", "tip", "checkin"]

archivos = [f.name for f in dbutils.fs.ls(RAW)]
print("Archivos en el Volume:", archivos)

faltantes = [e for e in ESPERADOS if not any(e in a for a in archivos)]
assert not faltantes, f"❌ Faltan archivos: {faltantes}. Suban la muestra al Volume."
print("✔ Los 5 datasets están presentes")

In [0]:
# DBTITLE 1: Exploración rápida de esquemas y volúmenes
for e in ESPERADOS:
    df = spark.read.json(f"{RAW}/{e}.json.gz")
    print(f"\n=== {e} · {df.count():,} filas ===")
    df.printSchema()


## Observaciones sobre los datos crudos

Las siguientes observaciones fueron identificadas durante la revisión de los archivos RAW y se utilizan como insumo para el modelado físico (E3) y la definición de controles de calidad de datos (E5).

### Observación 01 — Estructuras anidadas y campos complejos

Los archivos RAW de Yelp contienen atributos que no presentan una estructura completamente tabular.

Por ejemplo:

- `Business` contiene estructuras complejas como `attributes` y `hours`.
- `categories` contiene múltiples categorías asociadas a un mismo negocio.
- `User` contiene atributos multivalor como `friends` y `elite`.

**Implicancia para el modelado:**  
Bronze debe preservar la estructura original tanto como sea posible, mientras que Silver debe tipar, normalizar o seleccionar las estructuras necesarias según el modelo analítico.

**Implicancia para calidad:**  
Se deben validar estructuras nulas, vacías o con formatos no esperados sin interpretar automáticamente su ausencia como un error del negocio.

---

### Observación 02 — Fechas y tipos que requieren conformación explícita

Algunos atributos llegan desde RAW como texto o presentan formatos que requieren transformación antes de ser utilizados analíticamente.

Ejemplos:

- la fecha de `Review` debe convertirse a un timestamp válido;
- `yelping_since` de `User` requiere conversión explícita a fecha;
- `Check-in` contiene múltiples fechas asociadas a un `business_id`, por lo que debe transformarse desde su representación original a eventos individuales;
- campos numéricos como estrellas, conteos y métricas requieren tipos explícitos para evitar inferencias inconsistentes.

**Implicancia para el modelado:**  
Silver debe definir tipos de datos explícitos y un grano claro, especialmente para `slv_checkin_event`, cuyo grano corresponde a un negocio y un timestamp de check-in.

**Implicancia para calidad:**  
Deben existir reglas que validen fechas, rangos permitidos, valores numéricos y conversiones inválidas antes de publicar los datos conformados.

---

### Observación 03 — Integridad referencial, muestreo y duplicados requieren tratamientos diferentes

Los archivos RAW no deben interpretarse bajo el supuesto de que todas las entidades poseen cobertura completa entre sí.

En particular, el conjunto `User` utilizado en el proyecto corresponde a una muestra, por lo que existen `user_id` presentes en `Review` o `Tip` que no necesariamente están contenidos en la muestra de `User`.

Esta ausencia no debe clasificarse automáticamente como:

- registro inválido;
- error de integridad crítico;
- señal Trust & Safety;
- fraude potencial.

Adicionalmente, se identificaron duplicados técnicos en `Tip` que requieren una estrategia determinística de deduplicación durante la conformación.

**Implicancia para el modelado:**  
Las relaciones con `User` deben considerar que la cobertura es parcial y la deduplicación debe respetar el grano definido para cada entidad.

**Implicancia para calidad:**  
E5 debe distinguir entre fallas reales de calidad y limitaciones esperadas del dataset. Por ejemplo, la cobertura `Review → User` debe medirse y reportarse, pero no necesariamente bloquear el pipeline cuando la ausencia se explica por el muestreo.
```